# Load deps

In [1]:
# ! pip install -q torcheval

In [2]:
# # if src modules imported
# from google.colab import drive
# drive.mount('/content/drive')
import sys
app_path = '/content/drive/MyDrive/Projects/miniSD'
sys.path.append(app_path)

In [3]:
import shutil, os,torch
import matplotlib as mpl
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torch.nn.functional as F
from pathlib import Path
from glob import glob
from functools import partial
from torch.utils.data import DataLoader
from torch import nn,tensor, optim
from torch.optim import lr_scheduler
from torcheval.metrics import MulticlassAccuracy
from torchvision.io import read_image,ImageReadMode

from src.utils import set_seed, noop, urlsave
from src.datasets import DataLoaders, show_images, show_image
from src.training import get_dls
from src.learner import DeviceCB, ProgressCB, MetricsCB, Learner, TrainLearner, SingleBatchCB
from src.accel import MixedPrecision
from src.sgd import BatchSchedCB
from src.augment import RandErase
from src.init import GeneralRelu, BatchTransformCB, init_weights, clean_mem
from src.resnet import ResBlock

# Config

In [4]:
# os.environ['CUDA_VISIBLE_DEVICES']='2' --> not needed if we have a single GPU
torch.set_printoptions(precision=5, linewidth=140, sci_mode=False)
mpl.rcParams['figure.dpi'] = 70
set_seed(42)
bs = 512
epochs = 10

data_path = Path('artifacts/data')
data_path.mkdir(exist_ok=True, parents=True)
models_path = Path("artifacts/models")
models_path.mkdir(exist_ok=True, parents=True)

# Data processing

In [5]:
tin_path = data_path/'tiny-imagenet-200'
url = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
if not tin_path.exists():
    path_zip = urlsave(url, data_path)
    shutil.unpack_archive(path_zip, data_path)

In [6]:
class TinyDS:
    def __init__(self, path):
        self.path = Path(path)
        self.files = glob(str(path/'**/*.JPEG'), recursive=True)
    def __len__(self): return len(self.files)
    def __getitem__(self, i): return self.files[i],Path(self.files[i]).parent.parent.name

In [7]:
tds = TinyDS(tin_path/'train')
print(tds[0])
print(len(tds))

In [8]:
class TinyValDS(TinyDS):
    def __init__(self, path):
        super().__init__(path)
        path_anno = path / 'val_annotations.txt'
        self.anno = dict(o.split('\t')[:2] for o in path_anno.read_text().splitlines())
    def __getitem__(self, i):
        return self.files[i],self.anno[os.path.basename(self.files[i])]

In [9]:
vds = TinyValDS(tin_path/'val')
print(vds[0])

In [10]:
class TfmDS:
    def __init__(self, ds, tfmx=noop, tfmy=noop):
        self.ds,self.tfmx,self.tfmy = ds,tfmx,tfmy
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        x,y = self.ds[i]
        return self.tfmx(x),self.tfmy(y)

In [11]:
id2str = (tin_path/'wnids.txt').read_text().splitlines()
str2id = {v:k for k,v in enumerate(id2str)}

In [12]:
all_img_list = [
    read_image(x, mode=ImageReadMode.RGB)/255
    for x,y in tds
]
all_img_pt = torch.stack(all_img_list)

xmean = all_img_pt.mean(dim=(0,2,3))
xstd = all_img_pt.std(dim=(0,2,3))
print(f"xmean: {xmean} | xstd: {xstd}")

del all_img_list, all_img_pt

In [13]:
clean_mem()

In [14]:
def tfmx(x):
    img = read_image(x, mode=ImageReadMode.RGB)/255
    return (img-xmean[:,None,None])/xstd[:,None,None]
def tfmy(y): return tensor(str2id[y])
tfm_tds = TfmDS(tds, tfmx, tfmy)
tfm_vds = TfmDS(vds, tfmx, tfmy)

In [15]:
xi,yi = tfm_tds[0]
print(xi.mean(), xi.std(), xi.shape)
print(id2str[yi])

In [16]:
def denorm(x): return (x*xstd[:,None,None]+xmean[:,None,None]).clip(0,1)

In [17]:
dltrn = DataLoader(
    tfm_tds,
    batch_size=bs,
    shuffle=True,
    num_workers=os.cpu_count()
)
xb,yb = b = next(iter(dltrn))
show_image(denorm(xb[0]));
del dltrn

In [18]:
all_synsets = [o.split('\t') for o in (tin_path/'words.txt').read_text().splitlines()]
synsets = {k:v.split(',', maxsplit=1)[0] for k,v in all_synsets if k in id2str}

In [19]:
titles = [synsets[id2str[o]] for o in yb]
# print(', '.join(titles[:20]))
show_images(denorm(xb[:9]), titles=titles[:9], imsize=2.5);

In [20]:
dls = DataLoaders(
    *get_dls(tfm_tds, tfm_vds, bs=bs, num_workers=os.cpu_count())
)

del tfm_tds, tfm_vds, tds, vds

# Basic model

## Batch Transformations

In [21]:
def tfm_batch(b, tfm_x=noop, tfm_y = noop): return tfm_x(b[0]),tfm_y(b[1])

tfms = nn.Sequential(
    T.Pad(4),
    T.RandomCrop(64),
    T.RandomHorizontalFlip(),
    RandErase()
)
# this task (tinyimagenet) is a hard task and we won't succeed that much without augmentation
# the augmentations we use depends on the size and resolution of images.
# for example, for low-res images like our case, RandomResizedCrop doesn't work and returns a lot of blurred images
augcb = BatchTransformCB(partial(tfm_batch, tfm_x=tfms), on_val=False)

# batch-wise augmentation is faster than item-wise augmentation but it has some disadvantages:
# (the below points are only my perceptions. search for more reliable resources)
    # if a lot of augmentation is applied to all images of a single batch it can
        # move you away from the smooth parts of the loss curve
        # and toward distant parts of weights area
    # augmentation might also distort the meaning of images (e.g. turn 6 into 9 by flipping)
    # and if that is applied to all images in a batch uniformly, it can distort the distribution of the loss function

## Model Initial Arch

In [22]:
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)
nfs = (32,64,128,256,512,1024)

def get_dropmodel(act=act_gr, nfs=nfs, norm=nn.BatchNorm2d, drop=0.1):
    layers = [nn.Conv2d(3, nfs[0], 5, padding=2)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
               for i in range(len(nfs)-1)]
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(drop)]
    layers += [nn.Linear(nfs[-1], 200, bias=False), nn.BatchNorm1d(200)]
    return nn.Sequential(*layers).apply(iw)

In [23]:
# learn = TrainLearner(get_dropmodel(), dls, F.cross_entropy, cbs=[SingleBatchCB(), augcb, DeviceCB()])
# learn.fit(1)
# xb,yb = learn.batch
# show_images(denorm(xb.cpu())[:9], imsize=2.5);
# learn.summary()

In [24]:
opt_func = partial(optim.AdamW, eps=1e-5)

In [25]:
# lr_cbs = [DeviceCB(), augcb, MixedPrecision(), ProgressCB()]
# learn = Learner(get_dropmodel(), dls, F.cross_entropy, cbs=lr_cbs, opt_func=opt_func)
# learn.lr_find()

In [26]:
clean_mem()

In [27]:
# metrics = MetricsCB(accuracy=MulticlassAccuracy())
# cbs = [DeviceCB(), metrics, ProgressCB(plot=True), MixedPrecision()]
# lr = 0.1
# tmax = epochs * len(dls.train)
# sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
# xtra = [BatchSchedCB(sched), augcb]
# learn = Learner(get_dropmodel(), dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=opt_func)
# learn.fit(epochs)
# torch.save(learn.model, models_path / f'inettiny-basic-{epochs}_epochs.pkl')

# Deeper Model Arch

In [28]:
def res_blocks(n_bk, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
    return nn.Sequential(*[
        ResBlock(ni if i==0 else nf, nf, stride=stride if i==n_bk-1 else 1, ks=ks, act=act, norm=norm)
        for i in range(n_bk)])

nbks = (3,2,2,1,1)

def get_dropmodel(act=act_gr, nfs=nfs, nbks=nbks, norm=nn.BatchNorm2d, drop=0.2):
    layers = [ResBlock(3, nfs[0], ks=5, stride=1, act=act, norm=norm)]
    layers += [res_blocks(nbks[i], nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
               for i in range(len(nfs)-1)]
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(drop)]
    layers += [nn.Linear(nfs[-1], 200, bias=False), nn.BatchNorm1d(200)]
    return nn.Sequential(*layers).apply(iw)

In [29]:
# learn = TrainLearner(get_dropmodel(), dls, F.cross_entropy, cbs=[SingleBatchCB(), augcb, DeviceCB()])
# learn.fit(1)
# xb,yb = learn.batch
# # show_images(denorm(xb.cpu())[:9], imsize=2.5);
# learn.summary()

In [30]:
# opt_func = partial(optim.AdamW, eps=1e-5)
# learn = Learner(get_dropmodel(), dls, F.cross_entropy, cbs=lr_cbs, opt_func=opt_func)
# learn.lr_find()

In [31]:
clean_mem()

In [32]:
# metrics = MetricsCB(accuracy=MulticlassAccuracy())
# cbs = [DeviceCB(), metrics, ProgressCB(plot=True), MixedPrecision()]
# lr = 2e-1
# tmax = epochs * len(dls.train)
# sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
# xtra = [BatchSchedCB(sched), augcb]
# learn = Learner(get_dropmodel(), dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=opt_func)
# learn.fit(epochs)
# torch.save(learn.model, models_path / f'inettiny-custom-{epochs}_epochs.pkl')

# More augmentation (item-wise instead of batch-wise)

In [33]:
from PIL import Image

aug_tfms = nn.Sequential(
    T.Pad(4), T.RandomCrop(64),
    T.RandomHorizontalFlip(),
    T.TrivialAugmentWide() # https://arxiv.org/abs/2103.10158
)
norm_tfm = T.Normalize(xmean, xstd)
erase_tfm = RandErase() # expects normalized input, so we do it after normalization below

def tfmx(x, aug=False):
    x = Image.open(x).convert('RGB')
    if aug: x = aug_tfms(x)
    x = TF.to_tensor(x)
    x = norm_tfm(x)
    if aug: x = erase_tfm(x[None])[0]
    return x

In [34]:
tds = TinyDS(tin_path/'train')
vds = TinyValDS(tin_path/'val')

tfm_tds = TfmDS(tds, partial(tfmx, aug=True), tfmy)
tfm_vds = TfmDS(vds, tfmx, tfmy)

dls = DataLoaders(*get_dls(tfm_tds, tfm_vds, bs=bs, num_workers=os.cpu_count()))

del tfm_tds, tfm_vds, tds, vds

## Architecture improvement

- based on this [paper](https://arxiv.org/abs/1603.05027) from the same authors of the ResNet paper, instead of applying activation on the addition of resideual and main path (original arch in the paper Fig 1) we apply the activation only on the main path and apply a pure skip connection without activation applied on it (the proposed arch in the paper Fig 1)

In [35]:
def conv(ni, nf, ks=3, stride=1, act=nn.ReLU, norm=None, bias=True):
    layers = []
    if norm: layers.append(norm(ni))
    if act : layers.append(act())
    layers.append(nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2, bias=bias))
    return nn.Sequential(*layers)

def _conv_block(ni, nf, stride, act=act_gr, norm=None, ks=3):
    return nn.Sequential(conv(ni, nf, stride=1     , act=act, norm=norm, ks=ks),
                         conv(nf, nf, stride=stride, act=act, norm=norm, ks=ks))

class ResBlock(nn.Module):
    def __init__(self, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
        super().__init__()
        self.convs = _conv_block(ni, nf, stride, act=act, ks=ks, norm=norm)
        self.idconv = noop if ni==nf else conv(ni, nf, ks=1, stride=1, act=None, norm=norm)
        self.pool = noop if stride==1 else nn.AvgPool2d(2, ceil_mode=True)

    def forward(self, x): return self.convs(x) + self.idconv(self.pool(x)) # no act here!!!

def get_dropmodel(act=act_gr, nfs=nfs, nbks=nbks, norm=nn.BatchNorm2d, drop=0.2):
    layers = [
        nn.Conv2d(3, nfs[0], 5, padding=2)
        # can't start with resblock anymore because there's an act at the beginning of resblocks
    ]
    layers += [res_blocks(nbks[i], nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
               for i in range(len(nfs)-1)]
    layers += [
        act_gr(), norm(nfs[-1]), # act added here because resblocks don't end with act now
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(drop)
    ]
    layers += [nn.Linear(nfs[-1], 200, bias=False), nn.BatchNorm1d(200)]
    return nn.Sequential(*layers).apply(iw)

In [36]:
clean_mem()

In [37]:
def get_model(): return get_dropmodel(nbks=(4,3,3,2,1), drop=0.1)
# learn = TrainLearner(get_model(), dls, F.cross_entropy, cbs=[SingleBatchCB(), DeviceCB()])
# learn.fit(1)
# xb,yb = learn.batch
# show_images(denorm(xb.cpu())[:9], imsize=2.5);

In [38]:
clean_mem()

In [46]:
epochs = 20 # double the epochs
lr = 0.1
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
metrics = MetricsCB(accuracy=MulticlassAccuracy())
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), MixedPrecision()]
learn = Learner(get_model(), dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=opt_func)

In [47]:
learn.fit(epochs)

In [48]:
torch.save(learn.model, models_path / f'inettiny-trivaug-{epochs}_epochs.pkl')